# Customer Churn Forecasting - Model Deployment
## Module 9: Production Pipeline, Serialization & Real-Time Inference

Building an end-to-end Scikit-Learn Pipeline, serializing the trained model, and implementing a real-time churn prediction function for production deployment.

In [ ]:
# Step 1: Import Libraries
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import joblib

import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')

## 9.1 Load and Prepare the Dataset

In [ ]:
# Step 2: Load the Dataset
if os.path.exists('WA_Fn-UseC_-Telco-Customer-Churn.csv'):
    data = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
elif os.path.exists('telco_customer_churn.csv'):
    data = pd.read_csv('telco_customer_churn.csv')
else:
    raise FileNotFoundError('Dataset not found.')

# Preprocessing
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data['TotalCharges'].fillna(data['tenure'] * data['MonthlyCharges'], inplace=True)
data.drop(columns=['customerID'], inplace=True)
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})

# Feature engineering
data['AvgMonthlySpend'] = np.where(data['tenure'] > 0, data['TotalCharges'] / data['tenure'], data['MonthlyCharges'])
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
data['TotalServices'] = sum((data[col].isin(['Yes', 'Fiber optic', 'DSL'])).astype(int) for col in service_cols if col in data.columns)
data['ChargesPerService'] = np.where(data['TotalServices'] > 0, data['MonthlyCharges'] / data['TotalServices'], data['MonthlyCharges'])
contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
data['ContractRisk'] = data['Contract'].map(contract_risk)

# Encode categorical variables
cat_cols = data.select_dtypes(include=['object']).columns.tolist()
binary_cols = [col for col in cat_cols if data[col].nunique() == 2]
le = LabelEncoder()
for col in binary_cols:
    data[col] = le.fit_transform(data[col])
multi_cat_cols = [col for col in cat_cols if data[col].nunique() > 2]
data = pd.get_dummies(data, columns=multi_cat_cols, drop_first=True, dtype=int)

# Separate features and target
X = data.drop(columns=['Churn'])
y = data['Churn']

print(f'Dataset shape: {data.shape}')
print(f'Features: {X.shape[1]}')

## 9.2 Train-Test Split

In [ ]:
# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train samples: {len(X_train):,} | Test samples: {len(X_test):,}')
print(f'Train churn rate: {y_train.mean()*100:.2f}%')
print(f'Test churn rate:  {y_test.mean()*100:.2f}%')

## 9.3 Build Production Pipeline

In [ ]:
# Step 4: Create Pipeline and Train Model
# Using Logistic Regression pipeline for deployment (lightweight, interpretable)
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        solver='lbfgs', 
        max_iter=1000, 
        class_weight='balanced', 
        random_state=42,
        C=1.0
    ))
])

# Train the pipeline
model_pipeline.fit(X_train, y_train)

# Evaluate on test set
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

print('=' * 60)
print('PRODUCTION PIPELINE - TEST PERFORMANCE')
print('=' * 60)
print(f'\nAccuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'ROC-AUC:  {roc_auc_score(y_test, y_prob):.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Retained", "Churned"])}')

## 9.4 Save the Model

In [ ]:
# Step 5: Serialize the trained pipeline
model_filename = 'churn_prediction_model.pkl'
joblib.dump(model_pipeline, model_filename)

# Save feature names for inference validation
feature_names = X.columns.tolist()
joblib.dump(feature_names, 'feature_names.pkl')

file_size = os.path.getsize(model_filename) / 1024
print(f'Model saved: {model_filename} ({file_size:.1f} KB)')
print(f'Feature names saved: feature_names.pkl')
print(f'Number of features: {len(feature_names)}')

## 9.5 Load and Verify Model

In [ ]:
# Step 6: Load the saved model and verify
loaded_pipeline = joblib.load(model_filename)
loaded_features = joblib.load('feature_names.pkl')

# Verify predictions match
y_pred_loaded = loaded_pipeline.predict(X_test)
assert np.array_equal(y_pred, y_pred_loaded), 'Model predictions mismatch!'
print('Model loaded and verified successfully!')
print(f'Predictions match: True')

## 9.6 Real-Time Inference Function

In [ ]:
# Step 7: Reusable Real-Time Churn Prediction Function
def predict_churn(customer_data, threshold=0.50, model_path='churn_prediction_model.pkl'):
    """
    Predict customer churn probability and risk level.
    
    Parameters:
    -----------
    customer_data : pd.DataFrame
        Customer features (must match training feature set)
    threshold : float
        Decision threshold for churn classification (default: 0.50)
    model_path : str
        Path to the serialized model pipeline
        
    Returns:
    --------
    pd.DataFrame with predictions, probabilities, and risk levels
    """
    # Load model
    pipeline = joblib.load(model_path)
    
    # Predict
    probabilities = pipeline.predict_proba(customer_data)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    
    # Assign risk levels
    risk_levels = []
    for prob in probabilities:
        if prob >= 0.80:
            risk_levels.append('CRITICAL - Immediate Action Required')
        elif prob >= 0.60:
            risk_levels.append('HIGH - Retention Campaign Needed')
        elif prob >= 0.40:
            risk_levels.append('MODERATE - Monitor Closely')
        elif prob >= 0.20:
            risk_levels.append('LOW - Regular Engagement')
        else:
            risk_levels.append('MINIMAL - Loyal Customer')
    
    return pd.DataFrame({
        'Churn_Probability': np.round(probabilities, 4),
        'Churn_Prediction': predictions,
        'Risk_Level': risk_levels,
        'Action': ['Intervene' if p == 1 else 'Monitor' for p in predictions]
    })

print('predict_churn() function defined and ready for production use.')

## 9.7 Demo: Real-Time Prediction

In [ ]:
# Step 8: Demonstrate real-time prediction with test samples
print('=' * 70)
print('REAL-TIME CHURN PREDICTION DEMO')
print('=' * 70)

# Select 5 random test samples
sample_indices = np.random.RandomState(42).choice(len(X_test), size=5, replace=False)
sample_data = X_test.iloc[sample_indices]
sample_actual = y_test.iloc[sample_indices]

# Predict
results = predict_churn(sample_data)
results['Actual_Churn'] = sample_actual.values
results['Correct'] = (results['Churn_Prediction'] == results['Actual_Churn'])

print('\nSample Predictions:')
display(results)

print('\nPrediction Accuracy on Sample:', results['Correct'].mean() * 100, '%')

In [ ]:
# Step 9: Batch prediction on entire test set with risk distribution
batch_results = predict_churn(X_test)
risk_distribution = batch_results['Risk_Level'].value_counts()

print('\n' + '=' * 70)
print('BATCH PREDICTION - RISK DISTRIBUTION')
print('=' * 70)
print(f'\nTotal customers scored: {len(batch_results):,}')
print(f'\nRisk Level Distribution:')
for level, count in risk_distribution.items():
    pct = count / len(batch_results) * 100
    print(f'  {level}: {count:,} ({pct:.1f}%)')

print(f'\nCustomers requiring intervention: {batch_results["Churn_Prediction"].sum():,} ({batch_results["Churn_Prediction"].mean()*100:.1f}%)')

## 9.8 Deployment Summary

### Production Artifacts:
1. **`churn_prediction_model.pkl`** - Serialized Scikit-Learn Pipeline (Scaler + Logistic Regression)
2. **`feature_names.pkl`** - Feature names for input validation
3. **`predict_churn()`** - Ready-to-use inference function with risk level classification

### Integration Guide:
```python
import joblib
import pandas as pd

# Load model
model = joblib.load('churn_prediction_model.pkl')

# Predict on new customer data
new_customer = pd.DataFrame({...})  # Customer features
probability = model.predict_proba(new_customer)[:, 1]
prediction = model.predict(new_customer)
```

### Risk Level Classification:
| Probability Range | Risk Level | Recommended Action |
|---|---|---|
| >= 0.80 | CRITICAL | Immediate personal outreach, special offers |
| 0.60 - 0.79 | HIGH | Targeted retention campaign |
| 0.40 - 0.59 | MODERATE | Monitor closely, proactive engagement |
| 0.20 - 0.39 | LOW | Regular engagement programs |
| < 0.20 | MINIMAL | Standard customer experience |